In [49]:
import pandas as pd

PRQT_FILE_PATH = r"\\bosch.com\dfsrb\DfsDE\LOC\Rt\BST\09_projects\BMI420\External\02_Product_Development\03_System\04_Accel_System_Development\09_FT_Evaluations\BAI_CA\Possible_slope_correlation_CP3_off\acc_std_mean_CP2_CP3_CP4.parquet"
df_raw = pd.read_parquet(PRQT_FILE_PATH)

# DATA CLEANING AND PREPROC

In [50]:
# Filter Columns
df = df_raw.drop(columns=['prod_norm','prod_idx','lotid_norm', 'test_cod', 'waferid_norm' ])

# Create cold/hot hard_bin and soft_bin columns as target labels
cold = df[df['job_name'].str.contains('COLD', case=False, na=False)].drop_duplicates(
    subset=['wafer_id', 'x_coord', 'y_coord']
)[['wafer_id', 'x_coord', 'y_coord', 'hard_bin', 'soft_bin']].rename(
    columns={'hard_bin': 'cold_hard_bin', 'soft_bin': 'cold_soft_bin'}
)

hot = df[df['job_name'].str.contains('HOT', case=False, na=False)].drop_duplicates(
    subset=['wafer_id', 'x_coord', 'y_coord']
)[['wafer_id', 'x_coord', 'y_coord', 'hard_bin', 'soft_bin']].rename(
    columns={'hard_bin': 'hot_hard_bin', 'soft_bin': 'hot_soft_bin'}
)

df = df.merge(cold, on=['wafer_id', 'x_coord', 'y_coord'], how='left')
df = df.merge(hot, on=['wafer_id', 'x_coord', 'y_coord'], how='left')
df[['wafer_id', 'x_coord', 'y_coord', 'job_name', 'hard_bin', 'soft_bin',
    'cold_hard_bin', 'cold_soft_bin', 'hot_hard_bin', 'hot_soft_bin']].head(20)

# Pivot test_txt → test_result for CP2 rows only
cp2 = df[df['job_name'] == 'Herschel_CA_CP2_V2']
pivoted = cp2.pivot_table(
    index=['wafer_id', 'x_coord', 'y_coord'],
    columns='test_txt',
    values='test_result',
    aggfunc='first'
).reset_index()

# Merge in cold/hot bin columns (one row per die)
bins = df[['wafer_id', 'x_coord', 'y_coord',
           'cold_hard_bin', 'cold_soft_bin',
           'hot_hard_bin', 'hot_soft_bin']].drop_duplicates()

pivoted = pivoted.merge(bins, on=['wafer_id', 'x_coord', 'y_coord'], how='left')

# part_fail: True if either hard_bin or soft_bin is not 1
pivoted['cold_part_fail'] = ~((pivoted['cold_hard_bin'] == 1) & (pivoted['cold_soft_bin'] == 1))
pivoted['hot_part_fail'] = ~((pivoted['hot_hard_bin'] == 1) & (pivoted['hot_soft_bin'] == 1))
pivoted

,wafer_id,x_coord,y_coord,T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_MEAN_X[1],T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_MEAN_Y[1],T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_MEAN_Z[1],T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_SD_X[1],T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_SD_Y[1],T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_SD_Z[1],T17_62_FW_Combo_Sense_CBIST:F0_ACCGMN_MEAN_X[1],...,T17_65_FW_Combo_Noise_Remeas:ACCGMN_F7_HPM_3HOT_MEAN_Z[1],T17_65_FW_Combo_Noise_Remeas:ACCGMN_F7_HPM_3HOT_SD_X[1],T17_65_FW_Combo_Noise_Remeas:ACCGMN_F7_HPM_3HOT_SD_Y[1],T17_65_FW_Combo_Noise_Remeas:ACCGMN_F7_HPM_3HOT_SD_Z[1],cold_hard_bin,cold_soft_bin,hot_hard_bin,hot_soft_bin,cold_part_fail,hot_part_fail
0,DPK456-11-C0,1,50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,20.0,2033.0,20.0,2033.0,True,True
1,DPK456-11-C0,1,51,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,20.0,2033.0,20.0,2033.0,True,True
2,DPK456-11-C0,1,52,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,20.0,2033.0,20.0,2033.0,True,True
3,DPK456-11-C0,1,53,1102.0,694.0,-294.0,266.0,271.0,256.0,18.0,...,-1930.0,602.0,699.0,571.0,1.0,1.0,1.0,1.0,False,False
4,DPK456-11-C0,1,54,1249.0,-1068.0,-382.0,270.0,281.0,258.0,182.0,...,-2310.0,642.0,632.0,619.0,1.0,1.0,1.0,1.0,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51562,DPK456-17-B4,92,57,-375.0,-327.0,-255.0,277.0,264.0,263.0,-428.0,...,-1683.0,631.0,555.0,639.0,1.0,1.0,1.0,1.0,False,False
51563,DPK456-17-B4,92,58,-750.0,-1077.0,-1839.0,266.0,267.0,266.0,-708.0,...,-7217.0,666.0,623.0,585.0,1.0,1.0,1.0,1.0,False,False
51564,DPK456-17-B4,92,59,-775.0,-779.0,-868.0,267.0,278.0,279.0,-680.0,...,-3859.0,572.0,576.0,635.0,1.0,1.0,1.0,1.0,False,False
51565,DPK456-17-B4,92,60,-1595.0,62.0,-1250.0,270.0,297.0,263.0,-925.0,...,-5722.0,622.0,733.0,514.0,1.0,1.0,1.0,1.0,False,False


# PREDICTION

In [51]:
from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (classification_report, accuracy_score, confusion_matrix,
                              f1_score, precision_recall_curve, auc, average_precision_score)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from itertools import combinations
import numpy as np
import re

# --- Business cost parameters for threshold selection ---
# Missing a failing die (FN) is ~10x more costly than a false alarm (FP)
FN_COST = 10  # cost of missing a defective die
FP_COST = 1   # cost of false alarm (unnecessary retest)

RECALL_AT_PRECISION_TARGET = 0.8  # report recall at this precision level

targets = ['hot_part_fail', 'cold_part_fail']
features = [col for col in pivoted.columns if col not in ['wafer_id', 'cold_hard_bin', 'cold_soft_bin', 'hot_hard_bin', 'hot_soft_bin'] + targets]

# Drop rows with NaN in features or targets
data = pivoted.dropna(subset=features + targets).copy()

# Convert bin columns to categorical (nominal values, not numeric)
for t in targets:
    data[t] = data[t].astype(int).astype(str)

# Sanitize feature names for XGBoost (no [, ], or <)
clean_names = {col: re.sub(r'[\[\]<]', '_', col) for col in features}
data = data.rename(columns=clean_names)
features_clean = [clean_names[f] for f in features]

X = data[features_clean].values
groups = data['wafer_id'].values

# --- Stratified group split: enumerate wafer partitions, pick most balanced ---
unique_wafers = sorted(set(groups))
n_wafers = len(unique_wafers)
wafer_to_idx = {w: np.where(groups == w)[0] for w in unique_wafers}

# Compute per-wafer fail rate for each target
wafer_fail_rates = {}
for w in unique_wafers:
    idx = wafer_to_idx[w]
    rates = {}
    for t in targets:
        y_w = data[t].values[idx]
        rates[t] = (y_w == '1').mean()
    wafer_fail_rates[w] = rates

print("Per-wafer fail rates:")
for w in unique_wafers:
    rates_str = " | ".join(f"{t}: {wafer_fail_rates[w][t]:.3f}" for t in targets)
    print(f"  {w} ({len(wafer_to_idx[w])} dies): {rates_str}")

# Enumerate all train(~60%)/val(~20%)/holdout(~20%) partitions
# With 7 wafers: try all splits where train has 3-4 wafers, val and holdout each have 1-2+
best_split = None
best_score = float('inf')

for n_train in range(3, 5):  # 3 or 4 train wafers
    for train_wafers in combinations(unique_wafers, n_train):
        remaining = [w for w in unique_wafers if w not in train_wafers]
        n_rem = len(remaining)
        # Split remaining into val/holdout (at least 1 wafer each)
        for n_val in range(1, n_rem):
            for val_wafers in combinations(remaining, n_val):
                holdout_wafers = tuple(w for w in remaining if w not in val_wafers)
                if len(holdout_wafers) == 0:
                    continue

                # Compute fail rate per split per target
                score = 0.0
                for t in targets:
                    rates = []
                    for split_wafers in [train_wafers, val_wafers, holdout_wafers]:
                        all_idx = np.concatenate([wafer_to_idx[w] for w in split_wafers])
                        y_split = data[t].values[all_idx]
                        rates.append((y_split == '1').mean())
                    # Score: max deviation from overall mean across the 3 splits
                    score += max(rates) - min(rates)

                if score < best_score:
                    best_score = score
                    best_split = (train_wafers, val_wafers, holdout_wafers)

train_wafers, val_wafers, holdout_wafers = best_split
train_idx = np.concatenate([wafer_to_idx[w] for w in train_wafers])
val_idx = np.concatenate([wafer_to_idx[w] for w in val_wafers])
holdout_idx = np.concatenate([wafer_to_idx[w] for w in holdout_wafers])

print(f"\nStratified wafer-level 3-way split (minimized fail-rate imbalance):")
print(f"  Train:   {len(train_idx)} dies from {len(train_wafers)} wafers {list(train_wafers)}")
print(f"  Val:     {len(val_idx)} dies from {len(val_wafers)} wafers {list(val_wafers)}")
print(f"  Holdout: {len(holdout_idx)} dies from {len(holdout_wafers)} wafers {list(holdout_wafers)}")
for t in targets:
    tr = (data[t].values[train_idx] == '1').mean()
    vr = (data[t].values[val_idx] == '1').mean()
    hr = (data[t].values[holdout_idx] == '1').mean()
    print(f"  {t} fail rate — Train: {tr:.4f} | Val: {vr:.4f} | Holdout: {hr:.4f}")
assert len(set(groups[train_idx]) & set(groups[val_idx])) == 0, "Leakage: train/val overlap"
assert len(set(groups[train_idx]) & set(groups[holdout_idx])) == 0, "Leakage: train/holdout overlap"
assert len(set(groups[val_idx]) & set(groups[holdout_idx])) == 0, "Leakage: val/holdout overlap"

# Store results for dashboard
results = {}

for target in targets:
    print(f"\n{'='*60}")
    print(f"Target: {target}")
    print(f"{'='*60}")

    le = LabelEncoder()
    y_all = le.fit_transform(data[target].values)
    fail_class_idx = list(le.classes_).index('1')

    X_train, y_train = X[train_idx], y_all[train_idx]
    X_val, y_val = X[val_idx], y_all[val_idx]
    X_holdout, y_holdout = X[holdout_idx], y_all[holdout_idx]

    sample_weights = compute_sample_weight('balanced', y_train)

    model = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric='mlogloss',
    )
    model.fit(X_train, y_train, sample_weight=sample_weights)

    # --- Threshold selection on validation set using business cost ---
    y_val_proba = model.predict_proba(X_val)[:, fail_class_idx]

    # Sweep thresholds and compute business cost on validation set
    sweep_thresholds = np.arange(0.05, 0.96, 0.01)
    sweep_results = []
    for th in sweep_thresholds:
        y_pred_th = (y_val_proba >= th).astype(int)
        if fail_class_idx != 1:
            y_pred_th = 1 - y_pred_th
        cm_th = confusion_matrix(y_val, y_pred_th, labels=[0, 1])
        tn, fp, fn, tp = cm_th.ravel()
        cost = FN_COST * fn + FP_COST * fp
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        sweep_results.append({
            'threshold': th, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
            'recall': recall, 'precision': precision, 'FPR': fpr, 'cost': cost,
        })

    # Select threshold with minimum business cost
    best = min(sweep_results, key=lambda x: x['cost'])
    THRESHOLD = round(best['threshold'], 2)

    print(f"\n--- Threshold Selection (on validation set, FN_cost={FN_COST}, FP_cost={FP_COST}) ---")
    print(f"  Selected threshold: {THRESHOLD}")
    print(f"  Val cost: {best['cost']} (FN={best['FN']}, FP={best['FP']})")
    print(f"  Val recall: {best['recall']:.3f} | Val precision: {best['precision']:.3f} | Val FPR: {best['FPR']:.3f}")

    # Apply selected threshold on validation set for reporting
    y_val_pred = (y_val_proba >= THRESHOLD).astype(int)
    if fail_class_idx != 1:
        y_val_pred = 1 - y_val_pred

    val_labels = sorted(set(y_val) | set(y_val_pred))
    val_names = [str(le.classes_[i]) for i in val_labels]
    print(f"\n--- Validation Set (threshold={THRESHOLD}) ---")
    print(classification_report(y_val, y_val_pred, labels=val_labels, target_names=val_names))

    # PR curve from validation set (used for threshold sweep visualization)
    pr_precision, pr_recall, pr_thresholds = precision_recall_curve(y_val, y_val_proba, pos_label=fail_class_idx)

    # --- Holdout set: final unbiased evaluation with val-selected threshold ---
    y_hold_proba = model.predict_proba(X_holdout)[:, fail_class_idx]
    y_hold_pred = (y_hold_proba >= THRESHOLD).astype(int)
    if fail_class_idx != 1:
        y_hold_pred = 1 - y_hold_pred

    hold_labels = sorted(set(y_holdout) | set(y_hold_pred))
    hold_names = [str(le.classes_[i]) for i in hold_labels]

    acc = accuracy_score(y_holdout, y_hold_pred)
    f1_macro = f1_score(y_holdout, y_hold_pred, average='macro', labels=hold_labels)
    f1_weighted = f1_score(y_holdout, y_hold_pred, average='weighted', labels=hold_labels)
    cm = confusion_matrix(y_holdout, y_hold_pred, labels=hold_labels)
    report = classification_report(y_holdout, y_hold_pred, labels=hold_labels,
                                   target_names=hold_names, output_dict=True)

    # --- Business cost on holdout ---
    cm_hold_binary = confusion_matrix(y_holdout, y_hold_pred, labels=[0, 1])
    hold_tn, hold_fp, hold_fn, hold_tp = cm_hold_binary.ravel()
    holdout_cost = int(FN_COST * hold_fn + FP_COST * hold_fp)

    # --- PR-AUC and recall@precision on holdout ---
    hold_pr_precision, hold_pr_recall, hold_pr_thresholds = precision_recall_curve(
        y_holdout, y_hold_proba, pos_label=fail_class_idx)
    pr_auc = auc(hold_pr_recall, hold_pr_precision)

    # Recall at fixed precision: find max recall where precision >= target
    mask_prec = hold_pr_precision >= RECALL_AT_PRECISION_TARGET
    recall_at_prec = float(hold_pr_recall[mask_prec].max()) if mask_prec.any() else 0.0

    print(f"\n--- Holdout Set (final evaluation, threshold={THRESHOLD}) ---")
    print(f"Accuracy: {acc:.4f} | F1 macro: {f1_macro:.4f} | F1 weighted: {f1_weighted:.4f}")
    print(f"PR-AUC: {pr_auc:.4f} | Recall@{RECALL_AT_PRECISION_TARGET}prec: {recall_at_prec:.4f}")
    print(f"Business Cost: {holdout_cost}  (FN={hold_fn} × {FN_COST} + FP={hold_fp} × {FP_COST})")
    print(classification_report(y_holdout, y_hold_pred, labels=hold_labels, target_names=hold_names))

    results[target] = {
        'model': model,
        'le': le,
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'confusion_matrix': cm,
        'class_names': hold_names,
        'report': report,
        'feature_importances': model.feature_importances_,
        'pr_precision': pr_precision,
        'pr_recall': pr_recall,
        'pr_thresholds': pr_thresholds,
        'hold_pr_precision': hold_pr_precision,
        'hold_pr_recall': hold_pr_recall,
        'hold_pr_thresholds': hold_pr_thresholds,
        'pr_auc': pr_auc,
        'recall_at_prec': recall_at_prec,
        'recall_at_prec_target': RECALL_AT_PRECISION_TARGET,
        'threshold': THRESHOLD,
        'sweep_results': sweep_results,
        'fn_cost': FN_COST,
        'fp_cost': FP_COST,
        'holdout_cost': holdout_cost,
        'holdout_fn': int(hold_fn),
        'holdout_fp': int(hold_fp),
        'val_cost': int(best['cost']),
    }

print(f"\nAll models trained. Thresholds selected on validation set (business cost: FN×{FN_COST} + FP×{FP_COST}).")
print(f"Holdout evaluated with val-selected thresholds (no leakage).")

Per-wafer fail rates:
  DPK456-11-C0 (7366 dies): hot_part_fail: 0.042 | cold_part_fail: 0.044
  DPK456-12-E3 (7372 dies): hot_part_fail: 0.032 | cold_part_fail: 0.036
  DPK456-13-G6 (7361 dies): hot_part_fail: 0.050 | cold_part_fail: 0.054
  DPK456-14-B6 (7356 dies): hot_part_fail: 0.033 | cold_part_fail: 0.038
  DPK456-15-E1 (7361 dies): hot_part_fail: 0.050 | cold_part_fail: 0.053
  DPK456-16-G4 (7384 dies): hot_part_fail: 0.034 | cold_part_fail: 0.037
  DPK456-17-B4 (7367 dies): hot_part_fail: 0.039 | cold_part_fail: 0.042

Stratified wafer-level 3-way split (minimized fail-rate imbalance):
  Train:   22101 dies from 3 wafers ['DPK456-13-G6', 'DPK456-14-B6', 'DPK456-16-G4']
  Val:     14733 dies from 2 wafers ['DPK456-11-C0', 'DPK456-17-B4']
  Holdout: 14733 dies from 2 wafers ['DPK456-12-E3', 'DPK456-15-E1']
  hot_part_fail fail rate — Train: 0.0390 | Val: 0.0400 | Holdout: 0.0411
  cold_part_fail fail rate — Train: 0.0426 | Val: 0.0432 | Holdout: 0.0445

Target: hot_part_fail

--

In [53]:
from dash import Dash, html, dcc, Input, Output
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

app = Dash(__name__)

# --- Precompute figures ---

# 1) Overview bar chart (now includes PR-AUC)
overview_df = pd.DataFrame([
    {'Target': t, 'Accuracy': r['accuracy'], 'F1 Macro': r['f1_macro'],
     'F1 Weighted': r['f1_weighted'], 'PR-AUC': r['pr_auc']}
    for t, r in results.items()
])

fig_overview = go.Figure()
for metric, color in [('Accuracy', '#636EFA'), ('F1 Macro', '#EF553B'),
                       ('F1 Weighted', '#00CC96'), ('PR-AUC', '#AB63FA')]:
    fig_overview.add_trace(go.Bar(
        name=metric, x=overview_df['Target'], y=overview_df[metric],
        text=overview_df[metric].round(3), textposition='outside',
        marker_color=color
    ))
fig_overview.update_layout(
    barmode='group', title='Model Performance Overview',
    yaxis=dict(range=[0, 1.1], title='Score'), xaxis_title='Target',
    template='plotly_white', height=400, legend=dict(orientation='h', y=1.12)
)

# 2) Confusion matrices
cm_figs = {}
for t, r in results.items():
    cm = r['confusion_matrix']
    names = r['class_names']
    fig_cm = go.Figure(data=go.Heatmap(
        z=cm, x=names, y=names,
        colorscale='Blues', text=cm, texttemplate='%{text}',
        hovertemplate='True: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>'
    ))
    fig_cm.update_layout(
        title=f'Confusion Matrix — {t}', xaxis_title='Predicted', yaxis_title='Actual',
        template='plotly_white', height=450, yaxis=dict(autorange='reversed')
    )
    cm_figs[t] = fig_cm

# 3) Feature importance (top 20 per target)
fi_figs = {}
for t, r in results.items():
    imp = r['feature_importances']
    top_idx = np.argsort(imp)[-20:]
    fi_df = pd.DataFrame({
        'Feature': [features_clean[i] for i in top_idx],
        'Importance': imp[top_idx]
    }).sort_values('Importance')
    fig_fi = px.bar(fi_df, x='Importance', y='Feature', orientation='h',
                    color='Importance', color_continuous_scale='Viridis')
    fig_fi.update_layout(
        title=f'Top 20 Feature Importances — {t}',
        template='plotly_white', height=500, showlegend=False
    )
    fi_figs[t] = fig_fi

# 4) Precision-Recall curves (holdout + PR-AUC annotation + recall@precision marker)
pr_figs = {}
for t, r in results.items():
    fig_pr = go.Figure()
    # Holdout PR curve
    fig_pr.add_trace(go.Scatter(
        x=r['hold_pr_recall'], y=r['hold_pr_precision'],
        mode='lines', line=dict(color='#636EFA', width=2),
        name=f'Holdout PR (AUC={r["pr_auc"]:.3f})',
        fill='tozeroy', fillcolor='rgba(99,110,250,0.1)'
    ))
    # Mark the chosen threshold on holdout curve
    th = r['threshold']
    th_idx = np.argmin(np.abs(r['hold_pr_thresholds'] - th))
    fig_pr.add_trace(go.Scatter(
        x=[r['hold_pr_recall'][th_idx]], y=[r['hold_pr_precision'][th_idx]],
        mode='markers', marker=dict(size=14, color='red', symbol='x'),
        name=f'Threshold={th}'
    ))
    # Mark recall@precision point
    prec_target = r['recall_at_prec_target']
    recall_at = r['recall_at_prec']
    if recall_at > 0:
        fig_pr.add_trace(go.Scatter(
            x=[recall_at], y=[prec_target],
            mode='markers+text', marker=dict(size=12, color='#2ecc71', symbol='diamond'),
            name=f'Recall@{prec_target}prec={recall_at:.3f}',
            text=[f'  {recall_at:.3f}'], textposition='middle right',
            textfont=dict(size=12, color='#2ecc71')
        ))
        # Horizontal reference line at precision target
        fig_pr.add_hline(y=prec_target, line_dash='dash', line_color='#2ecc71',
                         opacity=0.5, annotation_text=f'Precision={prec_target}',
                         annotation_position='top left')
    fig_pr.update_layout(
        title=f'Precision-Recall Curve (Holdout, Fail class) — {t}  |  PR-AUC = {r["pr_auc"]:.3f}',
        xaxis_title='Recall', yaxis_title='Precision',
        template='plotly_white', height=420,
        xaxis=dict(range=[0, 1.05]), yaxis=dict(range=[0, 1.05]),
    )
    pr_figs[t] = fig_pr

# 5) Threshold sweep cost curves
sweep_figs = {}
for t, r in results.items():
    sweep_df = pd.DataFrame(r['sweep_results'])
    fig_sweep = go.Figure()
    fig_sweep.add_trace(go.Scatter(
        x=sweep_df['threshold'], y=sweep_df['cost'],
        mode='lines', line=dict(color='#EF553B', width=2),
        name='Business Cost', yaxis='y1'
    ))
    fig_sweep.add_trace(go.Scatter(
        x=sweep_df['threshold'], y=sweep_df['recall'],
        mode='lines', line=dict(color='#00CC96', width=2, dash='dash'),
        name='Recall (Fail)', yaxis='y2'
    ))
    fig_sweep.add_trace(go.Scatter(
        x=sweep_df['threshold'], y=sweep_df['precision'],
        mode='lines', line=dict(color='#636EFA', width=2, dash='dot'),
        name='Precision (Fail)', yaxis='y2'
    ))
    sel_th = r['threshold']
    sel_row = next(s for s in r['sweep_results'] if round(s['threshold'], 2) == sel_th)
    fig_sweep.add_trace(go.Scatter(
        x=[sel_th], y=[sel_row['cost']],
        mode='markers', marker=dict(size=14, color='red', symbol='star'),
        name=f'Selected (th={sel_th})', yaxis='y1'
    ))
    fig_sweep.update_layout(
        title=f'Threshold Sweep — {t} (FN×{r["fn_cost"]} + FP×{r["fp_cost"]})',
        xaxis_title='Decision Threshold',
        yaxis=dict(title='Business Cost', side='left', showgrid=True),
        yaxis2=dict(title='Recall / Precision', side='right', overlaying='y', range=[0, 1.05], showgrid=False),
        template='plotly_white', height=420,
        legend=dict(orientation='h', y=1.15),
    )
    sweep_figs[t] = fig_sweep

# 6) Per-class metrics tables
def make_report_table(report):
    rows = []
    for cls, metrics in report.items():
        if cls in ('accuracy', 'macro avg', 'weighted avg'):
            continue
        rows.append({
            'Class': cls,
            'Precision': f"{metrics['precision']:.3f}",
            'Recall': f"{metrics['recall']:.3f}",
            'F1-Score': f"{metrics['f1-score']:.3f}",
            'Support': int(metrics['support']),
        })
    for avg in ('macro avg', 'weighted avg'):
        if avg in report:
            m = report[avg]
            rows.append({
                'Class': avg,
                'Precision': f"{m['precision']:.3f}",
                'Recall': f"{m['recall']:.3f}",
                'F1-Score': f"{m['f1-score']:.3f}",
                'Support': int(m['support']),
            })
    return rows

# 7) Build sweep table rows
def make_sweep_table(sweep_results, selected_threshold):
    key_ths = set(range(0, len(sweep_results), 5))
    for i, s in enumerate(sweep_results):
        if round(s['threshold'], 2) == selected_threshold:
            key_ths.add(i)
    rows = []
    for i in sorted(key_ths):
        s = sweep_results[i]
        fnr = s['FN'] / (s['FN'] + s['TP']) if (s['FN'] + s['TP']) > 0 else 0
        rows.append({
            'Threshold': f"{s['threshold']:.2f}",
            'Recall': f"{s['recall']:.3f}",
            'Precision': f"{s['precision']:.3f}",
            'FPR': f"{s['FPR']:.3f}",
            'FNR': f"{fnr:.3f}",
            'Cost': int(s['cost']),
            '_selected': round(s['threshold'], 2) == selected_threshold,
        })
    return rows

# 8) Business cost KPI card helper
def make_cost_card(label, value, sub_text, color='#e74c3c'):
    return html.Div(style={
        'textAlign': 'center', 'padding': '15px 20px', 'borderRadius': '8px',
        'border': f'2px solid {color}', 'backgroundColor': '#fff',
        'minWidth': '150px',
    }, children=[
        html.Div(label, style={'fontSize': '12px', 'color': '#7f8c8d', 'marginBottom': '4px'}),
        html.Div(f"{value}" if isinstance(value, str) else f"{value:,}",
                 style={'fontSize': '26px', 'fontWeight': 'bold', 'color': color}),
        html.Div(sub_text, style={'fontSize': '11px', 'color': '#95a5a6', 'marginTop': '4px'}),
    ])

target_options = [{'label': t, 'value': t} for t in targets]

app.layout = html.Div(style={'fontFamily': 'Segoe UI, Arial, sans-serif', 'padding': '20px',
                              'backgroundColor': '#f8f9fa'}, children=[
    html.H1('XGBoost Model Performance Report',
            style={'textAlign': 'center', 'color': '#2c3e50', 'marginBottom': '5px'}),
    html.P('Predicting cold/hot part_fail from CP2 test features | Threshold selected by business cost on validation set',
           style={'textAlign': 'center', 'color': '#7f8c8d', 'marginBottom': '30px'}),

    # Overview section
    html.Div(style={'backgroundColor': 'white', 'borderRadius': '10px', 'padding': '20px',
                     'boxShadow': '0 2px 8px rgba(0,0,0,0.1)', 'marginBottom': '25px'}, children=[
        dcc.Graph(figure=fig_overview)
    ]),

    # Target selector
    html.Div(style={'backgroundColor': 'white', 'borderRadius': '10px', 'padding': '20px',
                     'boxShadow': '0 2px 8px rgba(0,0,0,0.1)', 'marginBottom': '25px'}, children=[
        html.Label('Select Target:', style={'fontWeight': 'bold', 'fontSize': '16px', 'marginBottom': '8px'}),
        dcc.Dropdown(id='target-select', options=target_options, value=targets[0],
                     clearable=False, style={'width': '350px'}),
    ]),

    # Business cost & PR-AUC KPI cards
    html.Div(id='cost-cards', style={'display': 'flex', 'gap': '15px', 'marginBottom': '25px',
                                      'justifyContent': 'center', 'flexWrap': 'wrap'}),

    # Confusion matrix + classification report side by side
    html.Div(style={'display': 'flex', 'gap': '20px', 'marginBottom': '25px'}, children=[
        html.Div(style={'flex': '1', 'backgroundColor': 'white', 'borderRadius': '10px', 'padding': '20px',
                         'boxShadow': '0 2px 8px rgba(0,0,0,0.1)'}, children=[
            dcc.Graph(id='confusion-matrix')
        ]),
        html.Div(style={'flex': '1', 'backgroundColor': 'white', 'borderRadius': '10px', 'padding': '20px',
                         'boxShadow': '0 2px 8px rgba(0,0,0,0.1)'}, children=[
            html.H3('Classification Report', style={'color': '#2c3e50', 'marginTop': '0'}),
            html.Div(id='report-table')
        ]),
    ]),

    # Precision-Recall curve
    html.Div(style={'backgroundColor': 'white', 'borderRadius': '10px', 'padding': '20px',
                     'boxShadow': '0 2px 8px rgba(0,0,0,0.1)', 'marginBottom': '25px'}, children=[
        dcc.Graph(id='pr-curve')
    ]),

    # Threshold sweep: cost curve + table side by side
    html.Div(style={'display': 'flex', 'gap': '20px', 'marginBottom': '25px'}, children=[
        html.Div(style={'flex': '1.2', 'backgroundColor': 'white', 'borderRadius': '10px', 'padding': '20px',
                         'boxShadow': '0 2px 8px rgba(0,0,0,0.1)'}, children=[
            dcc.Graph(id='sweep-cost-curve')
        ]),
        html.Div(style={'flex': '0.8', 'backgroundColor': 'white', 'borderRadius': '10px', 'padding': '20px',
                         'boxShadow': '0 2px 8px rgba(0,0,0,0.1)', 'overflowY': 'auto', 'maxHeight': '460px'}, children=[
            html.H3('Threshold Sweep Table', style={'color': '#2c3e50', 'marginTop': '0'}),
            html.Div(id='sweep-table')
        ]),
    ]),

    # Feature importance
    html.Div(style={'backgroundColor': 'white', 'borderRadius': '10px', 'padding': '20px',
                     'boxShadow': '0 2px 8px rgba(0,0,0,0.1)'}, children=[
        dcc.Graph(id='feature-importance')
    ]),
])

@app.callback(
    Output('confusion-matrix', 'figure'),
    Output('feature-importance', 'figure'),
    Output('report-table', 'children'),
    Output('pr-curve', 'figure'),
    Output('sweep-cost-curve', 'figure'),
    Output('sweep-table', 'children'),
    Output('cost-cards', 'children'),
    Input('target-select', 'value'),
)
def update_dashboard(selected_target):
    r = results[selected_target]
    cm_fig = cm_figs[selected_target]
    fi_fig = fi_figs[selected_target]
    pr_fig = pr_figs[selected_target]
    sweep_fig = sweep_figs[selected_target]

    # Classification report table
    rows = make_report_table(r['report'])
    header = html.Tr([html.Th(c, style={'padding': '10px 14px', 'borderBottom': '2px solid #dee2e6',
                                         'backgroundColor': '#f1f3f5', 'color': '#495057'})
                       for c in ['Class', 'Precision', 'Recall', 'F1-Score', 'Support']])
    body = [
        html.Tr([
            html.Td(row[c], style={
                'padding': '8px 14px', 'borderBottom': '1px solid #eee',
                'fontWeight': 'bold' if row['Class'] in ('macro avg', 'weighted avg') else 'normal',
                'backgroundColor': '#f8f9fa' if row['Class'] in ('macro avg', 'weighted avg') else 'white',
            }) for c in ['Class', 'Precision', 'Recall', 'F1-Score', 'Support']
        ]) for row in rows
    ]
    table = html.Table([html.Thead(header), html.Tbody(body)],
                       style={'width': '100%', 'borderCollapse': 'collapse', 'fontSize': '14px'})

    # Sweep table
    sweep_rows = make_sweep_table(r['sweep_results'], r['threshold'])
    sweep_cols = ['Threshold', 'Recall', 'Precision', 'FPR', 'FNR', 'Cost']
    sweep_header = html.Tr([html.Th(c, style={'padding': '8px 10px', 'borderBottom': '2px solid #dee2e6',
                                               'backgroundColor': '#f1f3f5', 'color': '#495057',
                                               'fontSize': '13px'})
                             for c in sweep_cols])
    sweep_body = [
        html.Tr([
            html.Td(row[c], style={
                'padding': '6px 10px', 'borderBottom': '1px solid #eee', 'fontSize': '13px',
                'fontWeight': 'bold' if row['_selected'] else 'normal',
                'backgroundColor': '#fff3cd' if row['_selected'] else 'white',
            }) for c in sweep_cols
        ]) for row in sweep_rows
    ]
    sweep_tbl = html.Table([html.Thead(sweep_header), html.Tbody(sweep_body)],
                           style={'width': '100%', 'borderCollapse': 'collapse'})

    # Business cost & ranking KPI cards
    prec_tgt = r['recall_at_prec_target']
    cost_cards = [
        make_cost_card('Holdout Cost', r['holdout_cost'],
                       f'FN={r["holdout_fn"]}×{r["fn_cost"]} + FP={r["holdout_fp"]}×{r["fp_cost"]}',
                       color='#e74c3c'),
        make_cost_card('PR-AUC', f"{r['pr_auc']:.3f}",
                       'Holdout, fail class',
                       color='#AB63FA'),
        make_cost_card(f'Recall@{prec_tgt}prec', f"{r['recall_at_prec']:.3f}",
                       f'Max recall at precision≥{prec_tgt}',
                       color='#2ecc71'),
        make_cost_card('Missed Fails (FN)', r['holdout_fn'],
                       f'Cost: {r["holdout_fn"] * r["fn_cost"]:,}',
                       color='#c0392b'),
        make_cost_card('False Alarms (FP)', r['holdout_fp'],
                       f'Cost: {r["holdout_fp"] * r["fp_cost"]:,}',
                       color='#f39c12'),
        make_cost_card('Threshold', f"{r['threshold']:.2f}",
                       'Selected on val set',
                       color='#2980b9'),
    ]

    return cm_fig, fi_fig, table, pr_fig, sweep_fig, sweep_tbl, cost_cards

app.run(jupyter_mode='inline', debug=False, port=8051)